In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [3]:
parsnp_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S1/parsnp_100_consensus/parsnp.vcf', delimiter='\t', skiprows=5)
ska_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S1/ska/alignment.vcf', delimiter='\t', skiprows=2)
# ska_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S1/ska/alignment_k15.vcf', delimiter='\t', skiprows=2)
# ska_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S1/ska/hiv_lo_snps.vcf', delimiter='\t', skiprows=1)

parsnp_data = parsnp_data.drop(columns=['ancestral_sequence.fa.ref'])

## add .fasta to all cols in ska_data, remove . instances or replace as 0
ska_data.columns = (
    list(ska_data.columns[:9]) +
    [f"{col}.fasta" for col in ska_data.columns[9:]]
)

ska_data = ska_data[ska_data['ALT'] != '.']
ska_data = ska_data.replace('.', 0)

/tmp/ipykernel_3497913/3292185192.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ska_data = ska_data.replace('.', 0)


In [4]:
parsnp_data[(parsnp_data["POS"]==5006) | (parsnp_data["POS"]==5010)]

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,24-10-0.056004.fasta,...,58-54-0.413551.fasta,18-30-0.060340.fasta,2-52-0.362219.fasta,22-48-0.304350.fasta,34-73-0.334718.fasta,36-97-0.371712.fasta,4-25-0.459532.fasta,10-83-0.076895.fasta,20-0-0.306216.fasta,80-3-0.454627.fasta
1061,Ancestral,5006,GTATGCAAGA.GTAAGTGATA,G,A,40,PASS,NaN,GT,0,...,1,0,0,0,0,0,0,0,0,1
1063,Ancestral,5010,GCAAGAGTAA.GTGATAGTTG,G,A,40,PASS,NaN,GT,0,...,1,0,0,0,1,0,0,0,0,1


In [5]:
test = parsnp_data.iloc[:,9:]
print((test==0).sum().sum())
print((test==1).sum().sum())
print((test==2).sum().sum())
print((test==3).sum().sum())
print((test==1).sum().sum() + (test==2).sum().sum() + (test==3).sum().sum())


69447
9398
767
51
10216


In [6]:
k = 21  # window size

def get_num_within_k(parsnp_data, k=21):
    pos = parsnp_data["POS"].to_numpy()
    sample_cols = parsnp_data.columns[9:]

    results = {}
    close_locations = {}  # store tuples of nearby variant positions

    num_variants_per_sample = []

    for col in sample_cols:
        idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
        num_variants_per_sample.append(len(idx))
        
        if len(idx) < 2:
            results[col] = 0
            close_locations[col] = []
            continue
        
        sample_pos = pos[idx]
        diffs = np.diff(sample_pos)
        
        threshold = k//2 + 1
        close_mask = diffs <= threshold
        
        # ---- collect tuples ----
        pairs = [
            (sample_pos[i], sample_pos[i+1])
            for i in np.where(close_mask)[0]
        ]
        close_locations[col] = pairs
        
        # ---- counting logic (your original method) ----
        count = (
            close_mask.sum() * 2
            - np.sum(close_mask[:-1] & close_mask[1:])
        )
        
        results[col] = count

    result_series = pd.Series(results)

    result_series
    result_series.sum()
    return result_series, num_variants_per_sample, close_locations

result_series, num_variants_per_sample, close_locations = get_num_within_k(parsnp_data, 21)
result_series_4, num_variants_per_sample_4, close_locations_4 = get_num_within_k(parsnp_data, 9)

In [7]:
print(max(num_variants_per_sample))
print(min(num_variants_per_sample))
print(np.median(num_variants_per_sample))

351
107
222.0


In [8]:
k = 21  # window size

pos = parsnp_data["POS"].to_numpy()
sample_cols = parsnp_data.columns[9:]

results = {}
cluster_locations = {}

for col in sample_cols:
    idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
    
    if len(idx) < 3:
        results[col] = 0
        cluster_locations[col] = []
        continue
    
    sample_pos = pos[idx]
    
    clusters = []
    counted = set()
    
    left = 0
    for right in range(len(sample_pos)):
        
        # shrink window if too large
        while sample_pos[right] - sample_pos[left] > k:
            left += 1
        
        window_size = right - left + 1
        
        if window_size >= 3:
            cluster = tuple(sample_pos[left:right+1])
            clusters.append(cluster)
            counted.update(cluster)
    
    cluster_locations[col] = clusters
    results[col] = len(counted)

result_series = pd.Series(results)

result_series
result_series.sum()

np.int64(2406)

In [9]:
cluster_locations

{'24-10-0.056004.fasta': [(np.int64(369), np.int64(370), np.int64(380)),
  (np.int64(369), np.int64(370), np.int64(380), np.int64(388)),
  (np.int64(544), np.int64(549), np.int64(551)),
  (np.int64(693), np.int64(699), np.int64(704)),
  (np.int64(699), np.int64(704), np.int64(717)),
  (np.int64(765), np.int64(771), np.int64(774)),
  (np.int64(823), np.int64(830), np.int64(831)),
  (np.int64(823), np.int64(830), np.int64(831), np.int64(836)),
  (np.int64(1062), np.int64(1078), np.int64(1082)),
  (np.int64(1078), np.int64(1082), np.int64(1085)),
  (np.int64(1941), np.int64(1949), np.int64(1956)),
  (np.int64(2267), np.int64(2273), np.int64(2276)),
  (np.int64(2267), np.int64(2273), np.int64(2276), np.int64(2284)),
  (np.int64(2430), np.int64(2431), np.int64(2433)),
  (np.int64(2430), np.int64(2431), np.int64(2433), np.int64(2445)),
  (np.int64(2433), np.int64(2445), np.int64(2453)),
  (np.int64(2445), np.int64(2453), np.int64(2459)),
  (np.int64(2477), np.int64(2483), np.int64(2489)),
  

In [10]:
ska_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,0-23-0.098532.fasta,...,64-65-0.413309.fasta,66-33-0.355178.fasta,68-66-0.019926.fasta,70-28-0.297632.fasta,72-67-0.016729.fasta,74-98-0.056818.fasta,76-55-0.131487.fasta,78-1-0.289326.fasta,80-3-0.454627.fasta,8-99-0.089017.fasta
20,Ancestral,21,0,C,T,0,0,0,GT,0,...,0,0,0,1,0,0,0,0,0,0
21,Ancestral,22,0,A,G,0,0,0,GT,1,...,0,0,0,0,0,0,0,0,0,0
47,Ancestral,48,0,A,"T,G",0,0,0,GT,0,...,0,2,2,0,0,0,0,0,0,0
48,Ancestral,54,0,T,C,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
49,Ancestral,61,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7156,Ancestral,9165,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,1,0,0
7160,Ancestral,9169,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
7163,Ancestral,9172,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
7166,Ancestral,9175,0,A,G,0,0,0,GT,1,...,0,0,0,0,0,0,0,0,0,0


In [11]:
parsnp_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,24-10-0.056004.fasta,...,58-54-0.413551.fasta,18-30-0.060340.fasta,2-52-0.362219.fasta,22-48-0.304350.fasta,34-73-0.334718.fasta,36-97-0.371712.fasta,4-25-0.459532.fasta,10-83-0.076895.fasta,20-0-0.306216.fasta,80-3-0.454627.fasta
0,Ancestral,10,AAAGCGTCTT.TTAAATGGCA,T,C,40,PASS,NaN,GT,0,...,1,0,0,0,0,0,0,0,0,0
1,Ancestral,14,GCGTCTTTAA.ATGGCACCAC,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,1,1,0,0,0,0
2,Ancestral,15,CGTCTTTAAA.TGGCACCACG,T,"C,A,G",40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,21,TAAATGGCAC.CACGGTGAAT,C,T,40,PASS,NaN,GT,0,...,0,0,0,0,1,1,0,0,0,0
4,Ancestral,22,AAATGGCACC.ACGGTGAATA,A,G,40,PASS,NaN,GT,0,...,0,0,1,0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1938,Ancestral,9165,GTCTGAAGTT.AGATAAGAAT,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
1939,Ancestral,9169,GAAGTTAGAT.AAGAATAGAG,A,G,40,PASS,NaN,GT,1,...,0,1,0,1,0,0,0,0,1,0
1940,Ancestral,9172,GTTAGATAAG.AATAGAGCCA,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,1,1,0,0,0,0
1941,Ancestral,9175,AGATAAGAAT.AGAGCCACGA,A,G,40,PASS,NaN,GT,0,...,0,0,1,0,0,0,1,0,0,0


In [12]:
len([col for col in parsnp_data.columns if col.endswith(".fasta")])

41

In [13]:
len([col for col in ska_data.columns if col.endswith(".fasta")])

41

In [14]:
def split_multiple_alt_alleles(df):

    df = df.copy()
    fasta_cols = df.columns[9:]  # sample columns

    multi_mask = df["ALT"].str.contains(",", na=False)

    # Separate multi-ALT rows
    df_multi = df[multi_mask].copy()
    df_single = df[~multi_mask].copy()

    new_rows = []

    for _, row in df_multi.iterrows():

        alts = row["ALT"].split(",")

        for i, alt in enumerate(alts, start=1):  # ALT index = 1-based

            new_row = row.copy()
            new_row["ALT"] = alt

            # Convert sample columns:
            # keep only values equal to this alt index
            for col in fasta_cols:
                new_row[col] = 1 if row[col] == i else 0

            new_rows.append(new_row)

    df_multi_split = pd.DataFrame(new_rows)

    # Combine back
    result = pd.concat([df_single, df_multi_split], ignore_index=True)

    return result

In [15]:
parsnp_data['POS'] = parsnp_data['POS'].astype(int)
ska_data['POS'] = ska_data['POS'].astype(int)

parsnp_data = split_multiple_alt_alleles(parsnp_data)
ska_data = split_multiple_alt_alleles(ska_data)

In [16]:
test = parsnp_data.iloc[:,9:]
print((test==0).sum().sum())
print((test==1).sum().sum())
print((test==2).sum().sum()) 

80271
10216
0


In [17]:
import os
import pandas as pd


def get_bronko_df(location):
    vcf_files = [f for f in os.listdir(location) if f.endswith(".vcf")]

    variant_dict = {}

    for vcf_file in vcf_files:
        sample_name = vcf_file.replace(".vcf", ".fasta")
        with open(os.path.join(location, vcf_file)) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                chrom, pos, _, ref, alt = fields[:5]
                key = (chrom, pos, ref, alt)
                if key not in variant_dict:
                    variant_dict[key] = {}
                variant_dict[key][sample_name] = 1
                
    print(variant_dict)

    # convert to DataFrame

    bronko = pd.DataFrame.from_dict(variant_dict, orient="index")
    bronko = bronko.fillna(0).astype(int)
    bronko.reset_index(inplace=True)
    bronko.rename(columns={"index": ["CHROM", "POS", "REF", "ALT"]}, inplace=True)

    new_names = ['#CHROM','POS','REF','ALT']
    bronko.columns = new_names + bronko.columns[4:].tolist()
    bronko['POS'] = bronko['POS'].astype(int)
    return bronko

bronko = get_bronko_df("/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S1/bronko_single_ref")  # change this to your VCF folder path
bronko

# df = bronko.drop(columns='index')

# # save table

# # df.to_csv("variant_presence_matrix.csv", index=False)
# print("Saved variant presence matrix to variant_presence_matrix.csv")


{('Ancestral', '61', 'A', 'G'): {'52-37-0.259740.fasta': 1, '56-61-0.198293.fasta': 1, '54-16-0.210629.fasta': 1, '62-88-0.463081.fasta': 1, '58-54-0.413551.fasta': 1, '60-78-0.305355.fasta': 1}, ('Ancestral', '117', 'G', 'T'): {'52-37-0.259740.fasta': 1, '32-77-0.273127.fasta': 1, '56-61-0.198293.fasta': 1, '36-97-0.371712.fasta': 1, '54-16-0.210629.fasta': 1, '64-65-0.413309.fasta': 1, '62-88-0.463081.fasta': 1, '58-54-0.413551.fasta': 1, '60-78-0.305355.fasta': 1, '34-73-0.334718.fasta': 1}, ('Ancestral', '141', 'G', 'A'): {'52-37-0.259740.fasta': 1}, ('Ancestral', '152', 'A', 'G'): {'52-37-0.259740.fasta': 1, '56-61-0.198293.fasta': 1, '74-98-0.056818.fasta': 1, '70-28-0.297632.fasta': 1, '54-16-0.210629.fasta': 1, '72-67-0.016729.fasta': 1, '64-65-0.413309.fasta': 1, '62-88-0.463081.fasta': 1, '78-1-0.289326.fasta': 1, '58-54-0.413551.fasta': 1, '66-33-0.355178.fasta': 1, '68-66-0.019926.fasta': 1, '60-78-0.305355.fasta': 1, '76-55-0.131487.fasta': 1, '80-3-0.454627.fasta': 1}, ('

,#CHROM,POS,REF,ALT,52-37-0.259740.fasta,56-61-0.198293.fasta,54-16-0.210629.fasta,62-88-0.463081.fasta,58-54-0.413551.fasta,60-78-0.305355.fasta,...,30-62-0.206087.fasta,4-25-0.459532.fasta,6-11-0.092686.fasta,28-5-0.191505.fasta,22-48-0.304350.fasta,2-52-0.362219.fasta,24-10-0.056004.fasta,0-23-0.098532.fasta,40-57-0.343879.fasta,44-89-0.015900.fasta
0,Ancestral,61,A,G,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,Ancestral,117,G,T,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,Ancestral,141,G,A,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,152,A,G,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Ancestral,173,G,A,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2084,Ancestral,8046,C,T,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2085,Ancestral,8523,A,G,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2086,Ancestral,8621,T,C,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2087,Ancestral,8996,A,G,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [18]:
parsnp_data.duplicated(["#CHROM", "POS", "REF", "ALT"]).sum()

np.int64(0)

In [19]:
def get_shared(parsnp_data, ska_data, bronko):
    df1 = parsnp_data
    df2 = ska_data
    df3 = bronko

    variant_cols = ["#CHROM", "POS", "REF", "ALT"]

    def add_suffix(df, suffix):
        sample_cols = [c for c in df.columns if c not in variant_cols]
        return df.rename(columns={c: f"{c}_{suffix}" for c in sample_cols})

    df1_ = add_suffix(df1, "1")
    df2_ = add_suffix(df2, "2")
    df3_ = add_suffix(df3, "3")

    merged = (
        df1_
        .merge(df2_, on=variant_cols, how="outer")
        .merge(df3_, on=variant_cols, how="outer")
    )
    

    fasta_cols = [c for c in merged.columns if ".fasta" in c]

    merged[fasta_cols] = (
        merged[fasta_cols]
        .fillna(0)
        .astype(int)
    )

    sample_cols = [c for c in df1.columns[9:] if c not in variant_cols]


    
    results = []
    ska_missed = []

    for c in sample_cols:
        col1 = f"{c}_1"
        col2 = f"{c}_2"
        col3 = f"{c}_3"

        a = merged[col1] == 1
        b = merged[col2] == 1
        d = merged[col3] == 1

        only_1 = (a & ~b & ~d).sum()
        only_2 = (~a & b & ~d).sum()
        only_3 = (~a & ~b & d).sum()

        shared_12_only = (a & b & ~d).sum()
        shared_13_only = (a & ~b & d).sum()
        shared_23_only = (~a & b & d).sum()

        shared_123 = (a & b & d).sum()
        
        if shared_13_only:
            print(c)
            ska_missed.append(c)

        results.append({
            "sample": c,
            "only_1": int(only_1),
            "only_2": int(only_2),
            "only_3": int(only_3),
            "shared_12_only": int(shared_12_only),
            "shared_13_only": int(shared_13_only),
            "shared_23_only": int(shared_23_only),
            "shared_123": int(shared_123)
        })

    overlap_df = pd.DataFrame(results)
    overlap_df
    return overlap_df, merged, ska_missed

In [20]:
overlap_df, merged, ska_missed = get_shared(parsnp_data, ska_data, bronko)
overlap_df

24-10-0.056004.fasta
14-53-0.427146.fasta
26-82-0.187728.fasta
74-98-0.056818.fasta
54-16-0.210629.fasta
52-37-0.259740.fasta
62-88-0.463081.fasta
28-5-0.191505.fasta
16-90-0.499308.fasta
66-33-0.355178.fasta
46-4-0.031722.fasta
6-11-0.092686.fasta
50-70-0.309395.fasta
40-57-0.343879.fasta
30-62-0.206087.fasta
70-28-0.297632.fasta
72-67-0.016729.fasta
56-61-0.198293.fasta
32-77-0.273127.fasta
48-79-0.278710.fasta
44-89-0.015900.fasta
64-65-0.413309.fasta
76-55-0.131487.fasta
60-78-0.305355.fasta
42-7-0.015329.fasta
38-12-0.232429.fasta
8-99-0.089017.fasta
68-66-0.019926.fasta
12-9-0.345404.fasta
0-23-0.098532.fasta
78-1-0.289326.fasta
58-54-0.413551.fasta
18-30-0.060340.fasta
2-52-0.362219.fasta
22-48-0.304350.fasta
34-73-0.334718.fasta
36-97-0.371712.fasta
4-25-0.459532.fasta
10-83-0.076895.fasta
20-0-0.306216.fasta
80-3-0.454627.fasta


,sample,only_1,only_2,only_3,shared_12_only,shared_13_only,shared_23_only,shared_123
0,24-10-0.056004.fasta,53,0,0,0,163,0,78
1,14-53-0.427146.fasta,67,0,0,0,188,0,92
2,26-82-0.187728.fasta,62,0,0,0,200,0,76
3,74-98-0.056818.fasta,5,0,0,0,49,0,98
4,54-16-0.210629.fasta,22,0,0,0,104,0,94
5,52-37-0.259740.fasta,27,0,0,0,89,0,104
6,62-88-0.463081.fasta,22,0,0,0,89,0,98
7,28-5-0.191505.fasta,30,0,0,0,164,0,69
8,16-90-0.499308.fasta,40,0,1,0,157,0,96
9,66-33-0.355178.fasta,10,0,1,0,62,0,90


In [21]:
overlap_df.iloc[:, 1:].sum()

only_1            1173
only_2               0
only_3              10
shared_12_only       2
shared_13_only    5418
shared_23_only       1
shared_123        3623
dtype: int64

In [22]:
def count_cluster_misses(merged, cluster_dict):

    miss_counts = {}

    for fname, clusters in cluster_dict.items():

        col1 = f"{fname}_1"  # parsnp
        col3 = f"{fname}_3"  # bronko

        if col1 not in merged.columns or col3 not in merged.columns:
            miss_counts[fname] = 0
            continue

        # flatten clusters → unique positions
        cluster_positions = set()
        for tup in clusters:
            cluster_positions.update(tup)

        if not cluster_positions:
            miss_counts[fname] = 0
            continue

        # filter merged to those positions
        subset = merged[merged["POS"].isin(cluster_positions)]

        # parsnp present but bronko absent
        misses = ((subset[col1] == 1) & (subset[col3] == 0)).sum()

        miss_counts[fname] = int(misses)

    return pd.Series(miss_counts)

out = count_cluster_misses(merged, close_locations_4)
out.sum()

np.int64(877)

In [23]:

out = count_cluster_misses(merged, close_locations)
out.sum()

np.int64(1048)

In [24]:
pos_set = set(pos for tup in close_locations_4["74-98-0.056818.fasta"] for pos in tup)
pos_set

{np.int64(4018),
 np.int64(4020),
 np.int64(5006),
 np.int64(5010),
 np.int64(7255),
 np.int64(7256),
 np.int64(7976),
 np.int64(7981),
 np.int64(8540),
 np.int64(8541)}

In [25]:
merged[(merged['74-98-0.056818.fasta_1'] == 1)  & (merged['POS'].isin(pos_set))][['POS', '74-98-0.056818.fasta_1', '74-98-0.056818.fasta_3']]

,POS,74-98-0.056818.fasta_1,74-98-0.056818.fasta_3
999,4018,1,0
1001,4020,1,0
1218,5006,1,0
1220,5010,1,1
1751,7255,1,1
1752,7256,1,1
1910,7976,1,1
1911,7981,1,1
2051,8540,1,1
2052,8541,1,1


In [26]:
out = count_cluster_misses(merged, cluster_locations)
out.sum()


np.int64(826)